# Report preprocessing figures

Generates the preprocessing figures used in the report from CSV files produced by the pipeline.

Outputs:
- Figure 1: final class distribution
- Figure 2: engineered feature correlation matrix
- Figure 3: engineered input feature groups

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pipeline.py").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find project root.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
FIGURES_DIR = PROJECT_ROOT / "src" / "preprocessing" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_CSV = DATA_DIR / "asl_features_engineered.csv"
df = pd.read_csv(FEATURES_CSV)

PROJECT_ROOT, FEATURES_CSV, FIGURES_DIR

## Figure 1

Final class distribution after cleaning and feature engineering.

In [ ]:
class_counts = (
    df["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="samples")
    .sort_values("label")
)

fig, ax = plt.subplots(figsize=(11, 4.8))
sns.barplot(data=class_counts, x="label", y="samples", color="#2F9C95", ax=ax)
ax.axhline(class_counts["samples"].mean(), color="#333333", linestyle="--", linewidth=1, label="Mean")
ax.set_title("Final Class Distribution After Cleaning and Feature Engineering")
ax.set_xlabel("ASL letter")
ax.set_ylabel("Samples")
ax.legend(frameon=False)
sns.despine()
plt.tight_layout()
output_path = FIGURES_DIR / "figure_1_final_class_distribution.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved to: {output_path}")

## Figure 2

Correlation matrix of derived engineered features, excluding raw landmark coordinates.

In [ ]:
def short_feature_label(feature: str) -> str:
    label = feature
    label = label.replace("_tip_distance", " dist.")
    label = label.replace("wrist_to_", "wrist-")
    label = label.replace("_tip", "")
    label = label.replace("_angle_", " angle " )
    label = label.replace("_vector_", " vector " )
    return label

metadata_cols = {"filepath", "label", "handedness"}
derived_features = [
    col for col in df.columns
    if col not in metadata_cols and not col.startswith("lm_")
]

corr = df[derived_features].corr(method="pearson")
corr = corr.rename(index=short_feature_label, columns=short_feature_label)

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(
    corr,
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax,
)
ax.set_title("Correlation Matrix of All Derived Engineered Features", fontsize=13)
plt.xticks(rotation=60, ha="right", fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
output_path = FIGURES_DIR / "figure_2_engineered_feature_correlation.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved to: {output_path}")

## Figure 3

Input feature groups derived from the final engineered CSV.

In [ ]:
columns = df.columns.tolist()
landmark_cols = [col for col in columns if col.startswith("lm_")]
distance_cols = [
    col for col in columns
    if col.endswith("_tip_distance") or (col.startswith("wrist_to_") and col.endswith("_tip"))
]
angle_cols = [col for col in columns if "_angle_" in col]
vector_cols = [col for col in columns if "_vector_" in col]

feature_counts = {
    "Normalized landmark\ncoordinates": len(landmark_cols),
    "Tip and wrist\ndistances": len(distance_cols),
    "Finger angles": len(angle_cols),
    "Finger direction\nvectors": len(vector_cols),
    "Handedness": int("handedness" in columns),
}

fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#4e79a7", "#59a14f", "#f28e2b", "#af7aa1", "#8c8c8c"]
bars = ax.bar(feature_counts.keys(), feature_counts.values(), color=colors, edgecolor="none")
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 1, f"{int(height)}", ha="center", va="bottom", fontsize=12)

ax.set_title("Engineered input feature groups", fontsize=16, pad=12)
ax.set_ylabel("Number of input features", fontsize=13)
ax.set_ylim(0, 75)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
output_path = FIGURES_DIR / "figure_3_engineered_feature_groups.png"
fig.savefig(output_path, dpi=300)
plt.show()
plt.close()
print(f"Saved to: {output_path}")